# Sabeel: round 2, 640 px vs 800 px

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 2 h). If Colab disconnects, **Run all** again: cell 4 continues where it stopped.

1. Settings and Drive  2. Install  3. Restore the dataset from Drive (same split as round 1)
4. Fine-tune the round-1 model at 800 px  5. Score both models on the same test images
6. CPU speed of both, with ONNX Runtime and OpenVINO  7. Comparison table, saved to `MyDrive/sabeel/round2_comparison.json`

In [ ]:
# 1. Settings, GPU check, Drive
BASE = "pothole"          # round 1 run (640 px), already trained
RUN = "pothole_800"       # this run
IMGSZ = 800
EPOCHS = 60               # fine-tuning from the round-1 model, not from scratch

import subprocess, os, glob, json, time
assert subprocess.run(["nvidia-smi"]).returncode == 0, "No GPU: Runtime → Change runtime type → T4 GPU"
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/sabeel"
assert os.path.exists(f"{SAVE}/runs/{BASE}/weights/best.pt"), "round-1 model not found on Drive"

In [ ]:
# 2. Install
!pip install -q -U ultralytics onnx onnxslim onnxruntime openvino

In [ ]:
# 3. Restore the dataset from Drive: identical train/val/test split to round 1, so the comparison is fair
CACHE = f"{SAVE}/data_cache.zip"
assert os.path.exists(CACHE), "No dataset cache on Drive: run cells 1-4 of sabeel_pothole_training.ipynb once"
if not os.path.exists("/content/data.yaml"):
    !unzip -q -o "{CACHE}" -d /content
print("dataset:", {s: len(glob.glob(f"/content/data/{s}/images/*")) for s in ("train", "val", "test")})

In [ ]:
# 4. Fine-tune the round-1 model at 800 px (resumes automatically after a disconnect)
import torch
from ultralytics import YOLO
LAST = f"{SAVE}/runs/{RUN}/weights/last.pt"
ckpt = torch.load(LAST, map_location="cpu", weights_only=False) if os.path.exists(LAST) else None
if ckpt is None:
    print("starting 800 px fine-tune from the round-1 model")
    YOLO(f"{SAVE}/runs/{BASE}/weights/best.pt").train(
        data="/content/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=-1, device=0,
        patience=20, cos_lr=True, close_mosaic=10, workers=2,
        project=f"{SAVE}/runs", name=RUN, exist_ok=True)
elif ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) >= 0:
    print(f"resuming at epoch {ckpt['epoch'] + 2} of {ckpt.get('train_args', {}).get('epochs', '?')}")
    YOLO(LAST).train(resume=True)
else:
    print("800 px training already finished")

In [ ]:
# 5. Score both models on the SAME test images; each gets its own confidence threshold, tuned on val
import numpy as np

def evaluate(run, imgsz):
    m = YOLO(f"{SAVE}/runs/{run}/weights/best.pt")
    v = m.val(data="/content/data.yaml", split="val", imgsz=imgsz, plots=False,
              project=f"{SAVE}/runs", name=f"{run}_val", exist_ok=True)
    f1 = np.asarray(v.box.f1_curve).mean(0)
    conf = round(float(np.asarray(v.box.px)[f1.argmax()]), 3)
    t = m.val(data="/content/data.yaml", split="test", imgsz=imgsz,
              project=f"{SAVE}/runs", name=f"{run}_test", exist_ok=True)
    tf1 = np.asarray(t.box.f1_curve).mean(0)
    idx = int(np.abs(np.asarray(t.box.px) - conf).argmin())
    return {"run": run, "imgsz": imgsz,
            "mAP50": round(float(t.box.map50), 4), "mAP50-95": round(float(t.box.map), 4),
            "precision": round(float(t.box.mp), 4), "recall": round(float(t.box.mr), 4),
            "conf_threshold": conf, "test_F1_at_threshold": round(float(tf1[idx]), 4)}

results = [evaluate(BASE, 640), evaluate(RUN, IMGSZ)]
for r in results: print(r)

In [ ]:
# 6. CPU speed: export both models to ONNX and OpenVINO, time each on CPU (same Colab CPU for all four)
import onnxruntime as ort, openvino as ov
core = ov.Core()

def bench(fn, x, n=30):
    for _ in range(3): fn(x)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(x); ts.append((time.perf_counter() - t0) * 1000)
    return round(float(np.median(ts)), 1)

for r in results:
    m = YOLO(f"{SAVE}/runs/{r['run']}/weights/best.pt")
    onnx_path = m.export(format="onnx", imgsz=r["imgsz"], opset=12, simplify=True, dynamic=False)
    ov_dir = m.export(format="openvino", imgsz=r["imgsz"], half=False, dynamic=False)
    x = np.random.rand(1, 3, r["imgsz"], r["imgsz"]).astype(np.float32)

    s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    name = s.get_inputs()[0].name
    r["cpu_ms_onnx"] = bench(lambda a: s.run(None, {name: a}), x)

    c = core.compile_model(glob.glob(f"{ov_dir}/*.xml")[0], "CPU")
    r["cpu_ms_openvino"] = bench(lambda a: c(a), x)
    r["onnx_output_shape"] = [int(d) for d in s.run(None, {name: x})[0].shape]
    print(r["run"], "| ONNX", r["cpu_ms_onnx"], "ms | OpenVINO", r["cpu_ms_openvino"], "ms")

In [ ]:
# 7. Comparison table, saved to Drive
a, b = results
print(f"{'':16}{'640 px (round 1)':>18}{'800 px (round 2)':>18}")
for k in ("mAP50", "mAP50-95", "precision", "recall", "test_F1_at_threshold", "conf_threshold", "cpu_ms_onnx", "cpu_ms_openvino"):
    print(f"{k:16}{a[k]:>18}{b[k]:>18}")
print(f"\n800 px vs 640 px: {b['mAP50'] - a['mAP50']:+.3f} mAP50 | "
      f"{b['cpu_ms_onnx'] / a['cpu_ms_onnx']:.2f}x CPU time (ONNX) | "
      f"800 px on OpenVINO = {b['cpu_ms_openvino'] / a['cpu_ms_onnx']:.2f}x today's 640 px on ONNX")
Path = __import__("pathlib").Path
Path(f"{SAVE}/round2_comparison.json").write_text(json.dumps(results, indent=2))
print(f"\nsaved {SAVE}/round2_comparison.json | models in {SAVE}/runs/{RUN}/weights/")

## Then
Save this notebook (**File → Download → Download .ipynb**) and send it, or just `MyDrive/sabeel/round2_comparison.json`.
If 800 px wins, also download `MyDrive/sabeel/runs/pothole_800/weights/best.onnx` (and the `best_openvino_model` folder if OpenVINO is clearly faster).